**LINEAR REGRESSION MODEL**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("train.csv")
train, test = train_test_split(df, test_size=0.2, random_state=42)

X_train = train[["GrLivArea"]]
y_train = train["SalePrice"]
X_test = test[["GrLivArea"]]
y_test = test["SalePrice"]

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Price per extra sq ft:", round(model.coef_[0], 1))
print("Starting point (intercept):", round(model.intercept_))
print("Average error (MAE):", round(mean_absolute_error(y_test, pred)))
print("R2 on test set:", round(r2_score(y_test, pred), 2))

Price per extra sq ft: 102.5
Starting point (intercept): 24900
Average error (MAE): 38341
R2 on test set: 0.55


*With living area alone, the model predicts about 102.5 more per square foot, and it’s off by about 38,300 on average (R² 0.55). Size explains about half of the variation in price, so the other half must come from other features.*

In [2]:
model.predict(pd.DataFrame({"GrLivArea": [2000]}))

array([229877.66599078])

*The model predicts about 229,878 for a 2,000 sq ft house. With a typical error of about 38,000, the real price could reasonably be anywhere from about 190,000 to 270,000.*

In [5]:
from sklearn.dummy import DummyRegressor

feature_sets = {
    "Size only": ["GrLivArea"],
    "Quality": ["GrLivArea", "OverallQual"],
    "+ Garage": ["GrLivArea", "OverallQual", "GarageCars"],
    "+ Year built": ["GrLivArea", "OverallQual", "GarageCars", "YearBuilt"],
    "+ Basement size": ["GrLivArea", "OverallQual", "GarageCars", "YearBuilt", "TotalBsmtSF"],
}

#Baseline: guess the average price for every house.
base = DummyRegressor(strategy= "mean").fit(train[["GrLivArea"]], train["SalePrice"])
base_pred = base.predict(test[["GrLivArea"]])
print("Baseline (always guess the average): MAE", round(mean_absolute_error(test["SalePrice"],base_pred)))
print()

for name, cols in feature_sets.items():
  m = LinearRegression().fit(train[cols], train["SalePrice"])
  p = m.predict(test[cols])
  print(name, "| MAE:", round(mean_absolute_error(test["SalePrice"], p)), "| R2:", round(r2_score(test["SalePrice"], p), 2))

Baseline (always guess the average): MAE 62576

Size only | MAE: 38341 | R2: 0.55
Quality | MAE: 28555 | R2: 0.75
+ Garage | MAE: 28122 | R2: 0.76
+ Year built | MAE: 27768 | R2: 0.76
+ Basement size | MAE: 25415 | R2: 0.79


*A baseline that guesses the average price for every house is off by about 62,600. Living area alone cuts that to about 38,300. Adding overall quality brings it to about 28,600, the biggest single improvement, and five features reach about 25,400 (R² 0.79), nearly 60% lower than the baseline. Garage size and year built added little because they overlap with size and quality.*